# 9.10.4.5 Interpretabilidad con LIME  &  9.10.4.6 Comparación de Resultados
### Lending Club Loan Data

> **Prerequisito:** Ejecutar primero `sklearn_lending_club.ipynb` y `pyspark_lending_club.ipynb`  
> para que existan `metrics_sklearn.json` y `metrics_spark.json` en el directorio de trabajo.

---

# Parte 1 — Interpretabilidad con LIME

LIME (*Local Interpretable Model-agnostic Explanations*) explica predicciones individuales de cualquier clasificador de caja negra generando un modelo lineal local alrededor de cada instancia.

## 1. Instalación y librerías

In [ ]:
!pip install lime -q
print('✅ LIME instalado')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json, warnings, time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import (accuracy_score, precision_score, recall_score,
                                     f1_score, roc_auc_score, roc_curve,
                                     confusion_matrix, ConfusionMatrixDisplay)
import lime
import lime.lime_tabular

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('✅ Librerías cargadas')

## 2. Reconstruir el pipeline sklearn

Se replica el pipeline completo del notebook sklearn para tener el modelo y los datos en memoria.  
Los mejores hiperparámetros se leen directamente de `metrics_sklearn.json`.

In [ ]:
# ── Carga ──────────────────────────────────────────────────────
df = pd.read_csv('df_clean.csv', low_memory=False)

FEATURES = [c for c in [
    'loan_amnt', 'int_rate', 'fico_range_high', 'emp_length',
    'annual_inc', 'purpose', 'home_ownership', 'dti',
    'term', 'sub_grade', 'verification_status',
    'open_acc', 'pub_rec', 'revol_util', 'mort_acc'
] if c in df.columns]

X = df[FEATURES].copy()
y = df['default'].copy()

NUM_COLS = X.select_dtypes(include='number').columns.tolist()
CAT_COLS = X.select_dtypes(include='object').columns.tolist()

# ── Muestra estratificada (igual que en sklearn notebook) ───────
X_s, _, y_s, _ = train_test_split(X, y, train_size=50_000,
                                   stratify=y, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_s, y_s, test_size=0.20, random_state=42, stratify=y_s)

# ── Codificación ─────────────────────────────────────────────────
label_cols = [c for c in ['sub_grade', 'term'] if c in CAT_COLS]
ohe_cols   = [c for c in CAT_COLS if c not in label_cols]

le_enc = {}
for col in label_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col]  = le.transform(X_test[col].astype(str))
    le_enc[col]  = le

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
Xtr_ohe = ohe.fit_transform(X_train[ohe_cols])
Xte_ohe = ohe.transform(X_test[ohe_cols])
ohe_names = ohe.get_feature_names_out(ohe_cols)

num_label = NUM_COLS + label_cols
X_train_enc = pd.concat([X_train[num_label].reset_index(drop=True),
                          pd.DataFrame(Xtr_ohe, columns=ohe_names)], axis=1)
X_test_enc  = pd.concat([X_test[num_label].reset_index(drop=True),
                          pd.DataFrame(Xte_ohe, columns=ohe_names)], axis=1)

scaler = StandardScaler()
X_train_enc[NUM_COLS] = scaler.fit_transform(X_train_enc[NUM_COLS])
X_test_enc[NUM_COLS]  = scaler.transform(X_test_enc[NUM_COLS])

# ── Cargar mejores hiperparámetros y re-entrenar ─────────────────
with open('metrics_sklearn.json') as f:
    m_sk = json.load(f)

t0 = time.time()
best_rf = RandomForestClassifier(
    n_estimators = m_sk['n_estimators'],
    max_depth    = m_sk['max_depth'],
    class_weight = 'balanced',
    random_state = 42, n_jobs=-1
)
best_rf.fit(X_train_enc, y_train)
t_train = time.time() - t0

y_pred       = best_rf.predict(X_test_enc)
y_pred_proba = best_rf.predict_proba(X_test_enc)[:, 1]

print(f'✅ Modelo reconstruido en {t_train:.2f} s')
print(f'   n_estimators={m_sk["n_estimators"]}  max_depth={m_sk["max_depth"]}')
print(f'   ROC AUC en test: {roc_auc_score(y_test, y_pred_proba):.4f}')

## 3. Selección de instancias mal clasificadas

Se eligen dos tipos de errores:
- **Falso Negativo (FN):** el modelo predijo *Fully Paid* pero el préstamo hizo default → el más costoso para el banco.
- **Falso Positivo (FP):** el modelo predijo *Charged Off* pero sí se pagó → cliente rechazado injustamente.

In [ ]:
y_test_r = y_test.reset_index(drop=True)
y_pred_s = pd.Series(y_pred)

fn_idx = y_test_r[(y_test_r == 1) & (y_pred_s == 0)].index.tolist()
fp_idx = y_test_r[(y_test_r == 0) & (y_pred_s == 1)].index.tolist()

print(f'Falsos Negativos (real=1, pred=0): {len(fn_idx):,}')
print(f'Falsos Positivos (real=0, pred=1): {len(fp_idx):,}')
print()
print(f'Instancia FN elegida → índice {fn_idx[0]}')
print(f'Instancia FP elegida → índice {fp_idx[0]}')

## 4. Configurar `LimeTabularExplainer`

In [ ]:
feature_names = X_train_enc.columns.tolist()

explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data         = X_train_enc.values,
    feature_names         = feature_names,
    class_names           = ['Fully Paid', 'Charged Off'],
    mode                  = 'classification',
    discretize_continuous = True,
    random_state          = 42
)

print('✅ LimeTabularExplainer listo')
print(f'   Features: {len(feature_names)}')
print(f'   Clases  : ["Fully Paid", "Charged Off"]')

## 5. Explicación — Falso Negativo (FN)

Esta instancia **era un default real** pero el modelo predijo que pagaría.  
Es el error más costoso para una institución financiera.

In [ ]:
inst_fn = X_test_enc.iloc[fn_idx[0]].values

exp_fn = explainer.explain_instance(
    data_row   = inst_fn,
    predict_fn = best_rf.predict_proba,
    num_features = 10,
    num_samples  = 1000
)

prob_fn = best_rf.predict_proba(inst_fn.reshape(1, -1))[0]
print(f'Clase real     : 1  (Charged Off / Default)')
print(f'Clase predicha : 0  (Fully Paid)')
print(f'P(Fully Paid)  : {prob_fn[0]:.4f}')
print(f'P(Default)     : {prob_fn[1]:.4f}')
print()
print('Top 10 variables más influyentes:')
print(f'{"Feature":<45} {"Contribución":>12}')
print('-' * 60)
for feat, w in exp_fn.as_list():
    flecha = "↑ sube default" if w > 0 else "↓ baja default"
    print(f'{feat:<45} {w:+.4f}  {flecha}')

In [ ]:
fig = exp_fn.as_pyplot_figure()
fig.set_size_inches(12, 6)
plt.title('LIME — Falso Negativo  (real=Default · pred=Fully Paid)',
          fontweight='bold', fontsize=12, pad=14)
plt.tight_layout()
plt.savefig('lime_01_falso_negativo.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Explicación — Falso Positivo (FP)

Esta instancia **sí pagó el préstamo** pero el modelo predijo default.  
Representa un cliente que sería rechazado injustamente.

In [ ]:
inst_fp = X_test_enc.iloc[fp_idx[0]].values

exp_fp = explainer.explain_instance(
    data_row   = inst_fp,
    predict_fn = best_rf.predict_proba,
    num_features = 10,
    num_samples  = 1000
)

prob_fp = best_rf.predict_proba(inst_fp.reshape(1, -1))[0]
print(f'Clase real     : 0  (Fully Paid)')
print(f'Clase predicha : 1  (Charged Off / Default)')
print(f'P(Fully Paid)  : {prob_fp[0]:.4f}')
print(f'P(Default)     : {prob_fp[1]:.4f}')
print()
print('Top 10 variables más influyentes:')
print(f'{"Feature":<45} {"Contribución":>12}')
print('-' * 60)
for feat, w in exp_fp.as_list():
    flecha = "↑ sube default" if w > 0 else "↓ baja default"
    print(f'{feat:<45} {w:+.4f}  {flecha}')

In [ ]:
fig = exp_fp.as_pyplot_figure()
fig.set_size_inches(12, 6)
plt.title('LIME — Falso Positivo  (real=Fully Paid · pred=Default)',
          fontweight='bold', fontsize=12, pad=14)
plt.tight_layout()
plt.savefig('lime_02_falso_positivo.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Comparación visual FN vs FP

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

palette = {'pos': '#e74c3c', 'neg': '#2980b9'}

for ax, exp, titulo, color in [
    (axes[0], exp_fn,
     'Falso Negativo\n(real=Default · pred=Fully Paid)', '#e74c3c'),
    (axes[1], exp_fp,
     'Falso Positivo\n(real=Fully Paid · pred=Default)', '#2980b9'),
]:
    items   = exp.as_list()
    feats   = [x[0] for x in items][::-1]
    weights = [x[1] for x in items][::-1]
    colors  = [color if w > 0 else '#bdc3c7' for w in weights]

    bars = ax.barh(range(len(feats)), weights, color=colors, edgecolor='white', height=0.7)
    ax.set_yticks(range(len(feats)))
    ax.set_yticklabels([f[:40] for f in feats], fontsize=8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(titulo, fontweight='bold', fontsize=11)
    ax.set_xlabel('Contribución LIME')

    for bar, w in zip(bars, weights):
        ax.text(w + (0.001 if w >= 0 else -0.001),
                bar.get_y() + bar.get_height()/2,
                f'{w:+.3f}', va='center',
                ha='left' if w >= 0 else 'right', fontsize=7)

plt.suptitle('Explicaciones LIME — Instancias Mal Clasificadas',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('lime_03_comparacion.png', bbox_inches='tight', dpi=150)
plt.show()

### Interpretación de resultados LIME

| Error | Causa probable | Impacto |
|-------|---------------|---------|
| **Falso Negativo** | Variables financieras "buenas" (FICO alto, int_rate bajo) enmascararon señales de riesgo | El banco pierde el capital prestado |
| **Falso Positivo** | Señales de riesgo aisladas (dti alto, annual_inc bajo) sobreactivaron la alarma | Un cliente solvente es rechazado injustamente |

LIME es especialmente valioso en finanzas porque regulaciones como **GDPR** y **Fair Credit Reporting Act** obligan a las instituciones a explicar individualmente cada decisión de crédito.

---
# Parte 2 — Comparación de Resultados: sklearn vs PySpark

## 8. Cargar métricas de ambos modelos

In [ ]:
with open('metrics_sklearn.json') as f:
    m_sk = json.load(f)
with open('metrics_spark.json') as f:
    m_sp = json.load(f)

# Normalizar clave ROC AUC (puede llamarse 'ROC_AUC' o 'ROC AUC')
for m in [m_sk, m_sp]:
    if 'ROC AUC' in m and 'ROC_AUC' not in m:
        m['ROC_AUC'] = m.pop('ROC AUC')

cols_show = ['Modelo', 'n_estimators', 'max_depth', 'Accuracy',
             'Precision', 'Recall', 'F1-score', 'ROC_AUC',
             'Tiempo_train_s', 'Tiempo_pred_s']

df_comp = pd.DataFrame([m_sk, m_sp])
cols_ok = [c for c in cols_show if c in df_comp.columns]

print('=' * 75)
print('              TABLA COMPARATIVA — sklearn vs PySpark')
print('=' * 75)
print(df_comp[cols_ok].to_string(index=False))
print('=' * 75)
df_comp[cols_ok]

## 9. Comparación de métricas

In [ ]:
metric_cols = [c for c in ['Accuracy','Precision','Recall','F1-score','ROC_AUC']
               if c in df_comp.columns]

x     = np.arange(len(metric_cols))
width = 0.35

vals_sk = df_comp.loc[df_comp['Modelo'].str.contains('sklearn'), metric_cols].values[0]
vals_sp = df_comp.loc[df_comp['Modelo'].str.contains('PySpark'), metric_cols].values[0]

fig, ax = plt.subplots(figsize=(12, 5))

bars1 = ax.bar(x - width/2, vals_sk, width,
               label='scikit-learn', color='#e74c3c', alpha=0.88, edgecolor='white')
bars2 = ax.bar(x + width/2, vals_sp, width,
               label='PySpark',      color='#8e44ad', alpha=0.88, edgecolor='white')

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.004,
            f'{h:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metric_cols, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Valor', fontsize=11)
ax.set_title('Comparación de Métricas — scikit-learn vs PySpark',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.25)

plt.tight_layout()
plt.savefig('comp_01_metricas.png', bbox_inches='tight', dpi=150)
plt.show()

## 10. Comparación de tiempos de cómputo

In [ ]:
tiempo_cols = [c for c in ['Tiempo_train_s', 'Tiempo_pred_s']
               if c in df_comp.columns]
modelos = df_comp['Modelo'].tolist()
titulos = ['Tiempo de Entrenamiento (s)', 'Tiempo de Predicción (s)']
colores = ['#e74c3c', '#8e44ad']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, titulo in zip(axes, tiempo_cols, titulos):
    vals = df_comp[col].values
    bars = ax.bar(modelos, vals, color=colores, edgecolor='white', width=0.45)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() * 1.03,
                f'{val:.2f} s', ha='center', va='bottom',
                fontweight='bold', fontsize=11)

    ax.set_title(titulo, fontweight='bold', fontsize=12)
    ax.set_ylabel('Segundos', fontsize=10)
    ax.tick_params(axis='x', rotation=10)
    ax.set_ylim(0, max(vals) * 1.25)

plt.suptitle('Comparación de Tiempos — scikit-learn vs PySpark',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comp_02_tiempos.png', bbox_inches='tight', dpi=150)
plt.show()

## 11. Curvas ROC superpuestas

La curva ROC completa se genera desde sklearn (tenemos las probabilidades en memoria).  
Para PySpark se muestra el AUC reportado como línea de referencia.

In [ ]:
fpr_sk, tpr_sk, _ = roc_curve(y_test, y_pred_proba)
auc_sk = m_sk['ROC_AUC']
auc_sp = m_sp['ROC_AUC']

fig, ax = plt.subplots(figsize=(9, 6))

# Curva sklearn
ax.plot(fpr_sk, tpr_sk, color='#e74c3c', lw=2.5,
        label=f'scikit-learn  (AUC = {auc_sk:.4f})')
ax.fill_between(fpr_sk, tpr_sk, alpha=0.07, color='#e74c3c')

# AUC PySpark como línea horizontal de referencia
ax.axhline(y=auc_sp, color='#8e44ad', lw=2, linestyle='--',
           label=f'PySpark  AUC = {auc_sp:.4f}  (valor reportado)')

ax.plot([0,1],[0,1],'k--', lw=1, alpha=0.4, label='Clasificador aleatorio')

ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('Curvas ROC — scikit-learn vs PySpark', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('comp_03_roc.png', bbox_inches='tight', dpi=150)
plt.show()

print('Nota: La curva completa de PySpark requiere convertir las predicciones')
print('a pandas dentro del notebook de Spark y calcular roc_curve allí.')

## 12. Gráfico radar — perfil de cada modelo

In [ ]:
from matplotlib.patches import FancyArrowPatch

metric_radar = [c for c in ['Accuracy','Precision','Recall','F1-score','ROC_AUC']
                if c in df_comp.columns]
N = len(metric_radar)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

vals_sk_r = df_comp.loc[df_comp['Modelo'].str.contains('sklearn'), metric_radar].values[0].tolist()
vals_sp_r = df_comp.loc[df_comp['Modelo'].str.contains('PySpark'),  metric_radar].values[0].tolist()
vals_sk_r += vals_sk_r[:1]
vals_sp_r += vals_sp_r[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

ax.plot(angles, vals_sk_r, 'o-', lw=2, color='#e74c3c', label='scikit-learn')
ax.fill(angles, vals_sk_r, alpha=0.12, color='#e74c3c')
ax.plot(angles, vals_sp_r, 's-', lw=2, color='#8e44ad', label='PySpark')
ax.fill(angles, vals_sp_r, alpha=0.12, color='#8e44ad')

ax.set_thetagrids(np.degrees(angles[:-1]), metric_radar, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Perfil de Métricas — sklearn vs PySpark',
             fontweight='bold', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)

plt.tight_layout()
plt.savefig('comp_04_radar.png', bbox_inches='tight', dpi=150)
plt.show()

## 13. Reflexión crítica

### ¿Qué entorno fue más rápido?
En este escenario (~50 000 filas en una sola máquina), **scikit-learn fue significativamente más rápido**.  
PySpark tiene un overhead inevitable: arranque de la JVM, serialización de datos entre Python y Java, y particionado.  
Ese costo solo se amortiza cuando el dataset supera la RAM disponible o se trabaja en un clúster real con decenas de nodos.

### ¿Cuál fue más preciso?
Ambos modelos deberían converger a métricas similares con los mismos hiperparámetros.  
Las diferencias menores (si existen) provienen del muestreo aleatorio y de cómo cada implementación construye los árboles internamente.

### ¿Cuándo elegir PySpark?
| Escenario | Recomendación |
|-----------|--------------|
| Dataset <100 K filas, máquina local | **scikit-learn** |
| Dataset >500 K – varios GB | **PySpark local o Databricks** |
| Clúster distribuido (Hadoop, EMR, Databricks) | **PySpark obligatorio** |
| Integración con pipelines de datos en tiempo real | **PySpark + Spark Streaming** |

### ¿Qué aporta LIME en producción?
- **Regulación:** GDPR Art. 22 y FCRA exigen explicaciones individuales para decisiones automatizadas de crédito.  
- **Auditoría de sesgos:** si LIME revela peso excesivo en variables como `addr_state` o `zip_code`, puede indicar discriminación geográfica.  
- **Confianza del analista:** los equipos de riesgo validan que el modelo use variables financieramente razonables antes del despliegue.

### Métrica principal: Recall
En detección de default, un **Falso Negativo** (predecir que paga pero hace default) implica pérdida del capital.  
Un **Falso Positivo** rechaza a un cliente solvente — impacto menor. Por eso **Recall** es la métrica de negocio prioritaria,  
complementada con ROC AUC para evaluar la capacidad de separación general del modelo.

## 14. Resumen final del proyecto

In [ ]:
print('=' * 68)
print('           RESUMEN FINAL — LENDING CLUB ML PROJECT')
print('=' * 68)

for label, m in [('scikit-learn', m_sk), ('PySpark     ', m_sp)]:
    auc_val = m.get('ROC_AUC', m.get('ROC AUC', 'N/A'))
    print(f'\n  {label}:')
    print(f'    Mejor modelo     : n_estimators={m["n_estimators"]:>3}  max_depth={m["max_depth"]}')
    print(f'    Accuracy         : {m["Accuracy"]}')
    print(f'    Recall           : {m["Recall"]}      ← métrica principal')
    print(f'    F1-score         : {m["F1-score"]}')
    print(f'    ROC AUC          : {auc_val}')
    print(f'    Tiempo train     : {m["Tiempo_train_s"]} s')
    if 'Tiempo_pred_s' in m:
        print(f'    Tiempo predicción: {m["Tiempo_pred_s"]} s')

print()
print('  LIME:')
print('    Instancias analizadas  : 1 Falso Negativo + 1 Falso Positivo')
print('    Variables más influyentes identificadas y graficadas')
print('    Explicaciones locales generadas para auditoría')
print()
print('  Conclusión:')
print('    scikit-learn = más rápido en escala local (<100K filas)')
print('    PySpark      = necesario en clúster o datos que no caben en RAM')
print('    LIME         = clave para cumplimiento regulatorio y confianza')
print('=' * 68)